# Experimentos

## Importando requisitos

In [ ]:
from typing import Any

import pandas
import os
import numpy
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

## Saindo do diretório notebooks

In [ ]:
import os
from pathlib import Path

os.chdir(Path(os.getcwd()).parent)
print(os.getcwd())

## Abrindo a planilha

In [ ]:
planilha = pandas.read_csv('../Dataset/DataSummary.csv')
planilha

## Filtrando na planilha os datasets selecionados

In [ ]:
planilha = planilha[planilha['ID'].between(97, 128)]
planilha

In [ ]:
def formatar_dataset(df: pandas.DataFrame) -> pandas.DataFrame:
    """
    Realiza a formatação do dataset.
    :param df: Dataset a ser formatado.
    :return: Dataset formatado.
    """
    classe = df.iloc[:, 0]
    series = df.iloc[:, 1:]

    df_novo = pandas.DataFrame({
        'classe': classe,
        'SérieTemporal': list(series.to_numpy())
    })

    return df_novo

In [ ]:
class OneNearestNeighbour:
    def __init__(self, algoritmo_distancia):
        self.algoritmo = algoritmo_distancia

    def fit(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train

    def predict(self, serie):
        melhor_distancia = float('inf')
        melhor_classe = None

        for xi_train, yi_train in zip(self.x_train, self.y_train):
            distancia = self.algoritmo.obter_distancia(serie, xi_train)

            if distancia < melhor_distancia:
                melhor_distancia = distancia
                melhor_classe = yi_train

        return melhor_classe

In [ ]:
def separar_x_y(dataset: pandas.DataFrame) -> tuple[list, list]:
    """
    Separa o conjunto de treino do conjunto de teste
    :param dataset: Dataset de entrada.
    :return: Tupla com os cunjuntos de treino e teste.
    """
    x = dataset['SérieTemporal'].tolist()
    y = dataset['classe'].tolist()

    return x, y

## Realizando experimentos no dataset da tabela toda

Definindo o tamanho máximo da recursão permitida

In [ ]:
from sklearn.metrics import accuracy_score
from app.model.DynamicTimeWarping import DynamicTimeWarping
from app.model.DerivativeDynamicTimeWarping import DerivativeDynamicTimeWarping
from app.model.LongestCommonSubsequence import LongestCommonSubsequence
from app.model.SoftDynamicTimeWarping import SoftDynamicTimeWarping


def treinar_modelo(planilha: pandas.DataFrame) -> tuple[dict[Any, Any], dict[Any, Any]]:
    """
    Treina modelos e obtém as métricas
    :param planilha: DataFrame contendo as séries temporais
    :return: Dicionário contendo as métricas de cada algoritmo
    """

    # Dicionário com as métricas dos algoritmos
    metricas = {}
    modelos = {}

    lista_nomes_datasets = planilha.iloc[:]['Name']
    for nome_dataset in lista_nomes_datasets:
        # Obtendo os caminhos do dataset de treino e teste
        print('Obtendo os caminhos do dataset de treino e teste')
        diretorio_dataset = os.path.join('../Dataset/UCRArchive_2018', nome_dataset)
        caminho_arquivo_treino = os.path.join(diretorio_dataset, '{}_TRAIN.tsv'.format(nome_dataset))
        caminho_arquivo_teste = os.path.join(diretorio_dataset, '{}_TEST.tsv'.format(nome_dataset))

        # Abrindo datasets
        print('Abrindo datasets')
        dataset_treino = pandas.read_csv(caminho_arquivo_treino, sep='\t', header=None)
        dataset_teste = pandas.read_csv(caminho_arquivo_teste, sep='\t', header=None)

        # Formatando datasets
        print('Formatando datasets')
        dataset_treino = formatar_dataset(dataset_treino)
        dataset_teste = formatar_dataset(dataset_teste)

        # Criando conjunto de treino e teste
        print('Criando conjunto de treino e teste')
        x_train, y_train = separar_x_y(dataset_treino)
        x_test, y_test = separar_x_y(dataset_teste)

        # Instanciando algoritmos
        print('Instanciando algoritmos')
        modelos[('dtw', nome_dataset)] = onn_dtw = OneNearestNeighbour(DynamicTimeWarping())
        modelos[('ddtw', nome_dataset)] = onn_ddtw = OneNearestNeighbour(DerivativeDynamicTimeWarping())
        modelos[('lcs', nome_dataset)] = onn_lcs = OneNearestNeighbour(LongestCommonSubsequence())
        modelos[('soft-dtw', nome_dataset)] = onn_soft_dtw = OneNearestNeighbour(SoftDynamicTimeWarping())

        # Treinando algoritmos
        print('Treinando algoritmos')
        onn_dtw.fit(x_train, y_train)
        onn_ddtw.fit(x_train, y_train)
        onn_lcs.fit(x_train, y_train)
        onn_soft_dtw.fit(x_train, y_train)

        # Testando algoritmos
        print('Testando algoritmos')
        y_dtw = [onn_dtw.predict(xi_test) for xi_test in x_test]
        y_ddtw = [onn_ddtw.predict(xi_test) for xi_test in x_test]
        y_lcs = [onn_lcs.predict(x_test) for xi_test in x_test]
        y_soft_dtw = [onn_soft_dtw.predict(x_test) for xi_test in x_test]

        # Obtendo acurácia
        print('Obtendo acurácia')
        acuracia_dtw = accuracy_score(y_test, y_dtw)
        acuracia_ddtw = accuracy_score(y_test, y_ddtw)
        acuracia_lcs = accuracy_score(y_test, y_lcs)
        acuracia_soft_dtw = accuracy_score(y_test, y_soft_dtw)
        print('Acurácia DTW: {}'.format(acuracia_dtw))
        print('Acurácia DDTW: {}'.format(acuracia_ddtw))
        print('Acurácia LCS: {}'.format(acuracia_lcs))
        print('Acurácia Soft DTW: {}'.format(acuracia_soft_dtw))

        # Salvando as métricas no dicionário
        print('Salvando as métricas no dicionário')
        if metricas[(nome_dataset, 'dtw')] is None:
            metricas[(nome_dataset, 'dtw')] = [acuracia_dtw]
        else:
            metricas[(nome_dataset, 'dtw')].append(acuracia_dtw)

        if metricas[(nome_dataset, 'ddtw')] is None:
            metricas[(nome_dataset, 'ddtw')] = [acuracia_ddtw]
        else:
            metricas[(nome_dataset, 'ddtw')].append(acuracia_ddtw)

        if metricas[(nome_dataset, 'lcs')] is None:
            metricas[(nome_dataset, 'lcs')] = [acuracia_lcs]
        else:
            metricas[(nome_dataset, 'lcs')].append(acuracia_lcs)

        if metricas[(nome_dataset, 'soft-dtw')] is None:
            metricas[(nome_dataset, 'soft-dtw')] = [acuracia_soft_dtw]
        else:
            metricas[(nome_dataset, 'soft-dtw')].append(acuracia_soft_dtw)

    return modelos, metricas

In [ ]:
modelos, metricas = treinar_modelo(planilha)
metricas

Salvando métricas

In [ ]:
import pickle
import pathlib

def salvar_arquivo_pickle(dados: object, filepath: str):
    """
    Salva um objeto como Pickle.
    :param dados: Objeto a ser salvo.
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """
    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'wb') as arquivo:
        pickle.dump(dados, arquivo)

In [ ]:
salvar_arquivo_pickle(metricas, 'metricas.pkl')